In [8]:
from gymnasium.core import ObsType
from typing import Literal, Any, SupportsFloat
from ppafm.io import loadXYZ
from ppafm.ocl.AFMulator import AFMulator
from ppafm.ocl.oclUtils import init_env
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.signal import argrelextrema
import numpy as np
from collections import defaultdict
import gymnasium as gym

class AFMEnvironment(gym.Env):
    metadata = {'render.modes': ['human', 'rgb_array']}
    
    def _compute_imgs(self, surface_path: str, params_path: str, i_platform: int = 0) -> tuple[AFMulator, np.ndarray]:
        """
        Generates AFM images with ppafm
        
        Parameters
        ----------
        surface_path : str
            Path to a .xyz file containing the surface
        params_path : str
            Path to a .ini file containing the parameters for the simulation
        i_platform : int
            Index of OpenCL device

        Returns
        -------
        AFMulator
            AFMulator object used to generate images
        np.ndarray
            Generated images. The second and third axis are already reversed.
        """
        init_env(i_platform=i_platform)
        xyzs, Zs, qs, _ = loadXYZ(surface_path)
        afmulator = AFMulator.from_params(params_path)
        afm_images = afmulator(xyzs, Zs, qs)
        
        return afmulator, afm_images[:,::-1,::-1]
    
    def __init__(self,
            surface_path: str,
            params_path: str,
            i_platform: int = 0,
            sigma: int = 4,
            num_historic_data: int = 4,
            render_mode: Literal[None, 'human', 'rgb'] = None
        ) -> None:
        """
        Constructor
        
        Parameters
        ----------
        surface_path : str
            Path to a .xyz file containing the surface
        params_path : str
            Path to a .ini file containing the parameters for the simulation
        i_platform : int
            Index of OpenCL device
        render_mode : Literal[None, 'human', 'rgb']
            Render mode to use
        """
        super().__init__()

        self.num_historic_data = num_historic_data
        
        # Generate images
        self.afmulator, self.afm_images = self._compute_imgs(surface_path, params_path, i_platform)
        
        # Calculate heights for each slice
        self.z_height_map = np.linspace(
            self.afmulator.scan_window[0][2],
            self.afmulator.scan_window[1][2] - self.afmulator.df_steps * self.afmulator.dz,
            self.afmulator.scan_dim[2] - self.afmulator.df_steps + 1,
        )
        
        # Get all minima in the z direction and only keep the highest one
        minima = np.array(argrelextrema(self.afm_images, np.less_equal, axis=2)).T
        minima_dict = defaultdict(list)
        for xx, yy, zz in minima:
            minima_dict[xx, yy].append(zz)
            minima_dict[xx, yy] = [max(minima_dict[xx, yy])]
        
        argmin_image = np.zeros(self.afm_images.shape[0:-1], dtype=int)
        for pixel, val in minima_dict.items():
            argmin_image[pixel] = int(val[0])
        
        # Convert index to actual height and smooth optimal surface
        self.min_image = self.z_height_map[argmin_image]
        self.optimal_height = gaussian_filter(self.min_image, sigma=sigma)
        #self.blurred_argmin_image = np.argmin(np.abs(blurred))
        
        # Define observation space
        x_px_max, y_px_max, _ = self.afm_images.shape
        self.z_min = self.afmulator.scan_window[0][2]
        self.z_max = self.afmulator.scan_window[1][2] - self.afmulator.df_steps * self.afmulator.dz
        self.observation_space = gym.spaces.Dict(
            {
                "x": gym.spaces.Box(0, x_px_max - 1, shape=(num_historic_data,), dtype=np.int64),
                "y": gym.spaces.Box(0, y_px_max - 1, shape=(num_historic_data,), dtype=np.int64),
                "dz": gym.spaces.Box(-self.z_max, self.z_max, shape=(num_historic_data,), dtype=np.float64),
                "df": gym.spaces.Box(-np.inf, np.inf, shape=(num_historic_data,), dtype=np.float64),
            }
        )
        
        # Define action space
        # TODO: What is the maximum speed for the tip?
        self.action_space = gym.spaces.Box(-1, 1, shape=(1,), dtype=np.float64)

        self.reset()

    def _get_closest_slice_index(self, z: float) -> np.int64:
        """
        Returns the index of the slice closest to a given z height

        Parameters
        ----------
        z : float
            Height of the desired slice

        Returns
        -------
        int : index of the closest slice
        """
        return np.abs(self.z_height_map - z).argmin()

    def _get_two_closest_z_planes(self, z):
        """
        Returns the indices of the two closest values in array to the given value.

        Parameters
        ----------
        value : float
            The target value

        Returns
        -------
        tuple : (int, int)
            Indices of the two closest values in ascending order
        """
        return np.sort(np.argsort(np.abs(self.z_height_map - z))[0:2])


    def _get_interpolated_df(self, x: int, y: int, z: float):
        """
        Interpolates a value at the given xy coordinates
        for a specific height `z` using two closest z-planes.

        Parameters
        ----------
        x : int
            The x index
        y : int
            The y index
        z : float
            The z position at which the value needs to be interpolated.

        Returns
        -------
        float
            The interpolated value at the specified z-coordinate.
        """
        z1, z2 = self._get_two_closest_z_planes(z)
        k = (self.afm_images[x, y, z2] - self.afm_images[x, y, z1]) / (self.z_height_map[z2] - self.z_height_map[z1])
        return k * (z - self.z_height_map[z1]) + self.afm_images[x, y, z1]


    def _get_obs(self):
        """
        Convert internal state to observation format.

        Returns
        -------
        dict
            Observation dictionary with x and y position, difference to the starting height and change in frequency
        """
        return {
            "x": self._x,
            "y": self._y,
            "dz": self._dz,
            "df": self._df
        }

    def _get_info(self):
        """
        Compute auxiliary information for debugging.

        Returns
        -------
            dict : z height and corresponding index
        """
        return {
            "z:": self.z_start,
            "z_i": self.z_start_index
        }

    # TODO: Randomize start position and rotation. Maybe also switch between different surfaces?
    def reset(self, seed: int | None = None, options : dict[str, Any] | None = None) -> tuple[ObsType, dict[str, Any]]:
        """
        Resets the environment
        
        Parameters
        ----------
        seed : int | None
            Seed for the random number generator
        options : dict[str, Any] | None
            Options for super class

        Returns
        -------

        """
        super().reset(seed=seed, options=options)

        if seed is not None:
            np.random.seed(seed)


        z = self.z_max - np.random.rand()
        self.z_start_index = self._get_closest_slice_index(z)
        self.z_start = self.z_height_map[self.z_start_index]

        self._x = np.zeros(self.num_historic_data, dtype=np.int64) # [0 for _ in range(self.num_historic_data)]
        self._y = np.zeros(self.num_historic_data, dtype=np.int64) # [0 for _ in range(self.num_historic_data)]
        # self.z_i = self.z_start_index * np.ones(self.num_historic_data) #[z_start_index for _ in range(self.num_historic_data)]
        # self.z = self.z_start * np.ones(self.num_historic_data) # [z for _ in range(self.num_historic_data)]
        self._dz = np.zeros(self.num_historic_data) # [0 for _ in range(self.num_historic_data)]
        self._df = self.afm_images[0, 0, self.z_start_index] * np.ones(self.num_historic_data) # [self.afm_images[0,0,z_start_index] for _ in range(self.num_historic_data)]

        return self._get_obs(), self._get_info()

    def _insert_into_array(self, array : np.ndarray, data_point):
        """
        Modifies an array by rolling it to the right and inserting a new data point at the beginning.

        Parameters
        ----------
        array : numpy.ndarray
            The input array to be modified.
        data_point : Any
            The new data point

        Returns
        -------
        numpy.ndarray
            The modified array
        """
        array = np.roll(array, 1)
        array[0] = data_point
        return array
        
    def step(self, action) -> tuple[ObsType, SupportsFloat, bool, bool, dict[str, Any]]:
        """Execute one timestep within the environment.

        Args:
            action: The action to take (0-3 for directions)

        Returns:
            tuple: (observation, reward, terminated, truncated, info)
        """
        x_px_max, y_px_max, _ = self.afm_images.shape
        x_current, y_current = int(self._x[0]), int(self._y[0])

        # Determine new position based on raster scan pattern
        if y_current % 2 == 0:  # Even rows: moving right
            if x_current + 1 >= x_px_max:
                # End of row reached, move to next row
                self._y = self._insert_into_array(self._y, y_current + 1)
            else:
                # Move right
                self._x = self._insert_into_array(self._x, x_current + 1)
        else:  # Odd rows: moving left
            if x_current - 1 < 0:
                # Beginning of row reached, move to next row
                self._y = self._insert_into_array(self._y, y_current + 1)
            else:
                # Move left
                self._x = self._insert_into_array(self._x, x_current - 1)

        x_new, y_new = int(self._x[0]), int(self._y[0])

        # Update dz and df
        dz_new = self._dz[0] + action[0]
        self._dz = self._insert_into_array(self._dz, dz_new)
        z_new = self.z_start + dz_new
        self._df = self._insert_into_array(self._df, self._get_interpolated_df(x_new, y_new, z_new))

        # Check for crashes
        # TODO: Add tolerance?
        if z_new < self.min_image[x_new, y_new]:
            return self._get_obs(), -100, True, False, self._get_info()

        # TODO: Base reward should be set by variable
        reward = 10 - self.optimal_height[x_new, y_new] + z_new

        terminated = False
        if y_new % 2 == 0 and y_new == y_px_max - 1 and x_new == x_px_max - 1:
            terminated = True
        elif y_new % 2 == 1 and y_new == 0 and x_new == 0:
            terminated = True

        return self._get_obs(), reward, terminated, False, self._get_info()
        

In [9]:
env = AFMEnvironment("materials/pt_111_small.xyz", "materials/params_code.ini")

Initializing an OpenCL environment on NVIDIA CUDA
sigma:  0.7
Spherical harmonic: s
sigma:  0.7
Spherical harmonic: s


In [6]:
env.reset(seed=123)
print(env._get_obs())
print(env.z_start_index)
print(env.z_start)
gym.spaces.Box(-np.inf, np.inf, shape=(1,), dtype=np.float64).sample()[0]

{'x': array([0, 0, 0, 0]), 'y': array([0, 0, 0, 0]), 'dz': array([0., 0., 0., 0.]), 'df': array([-0.08781265, -0.08781265, -0.08781265, -0.08781265])}
730
21.3


np.float64(0.31344616355732535)

In [10]:
from gymnasium.utils.env_checker import check_env

# This will catch many common issues
try:
    check_env(env)
    print("Environment passes all checks!")
except Exception as e:
    print(f"Environment has issues: {e}")

Environment passes all checks!


In [40]:
observation = env._get_obs()
print(observation["x"] in gym.spaces.Box(0, 200, shape=(4,), dtype=np.int64))
print(observation["y"] in gym.spaces.Box(0, 200, shape=(4,), dtype=np.int64))
print(observation["dz"] in gym.spaces.Box(-22., 22., shape=(4,), dtype=np.float64))
print(observation["df"] in gym.spaces.Box(-np.inf, np.inf, shape=(4,), dtype=np.float64))

False
False
True
True


In [15]:
np.argmin(np.abs(env.z_height_map-15))

np.int64(100)

In [15]:
env.z_height_map

array([14.  , 14.01, 14.02, 14.03, 14.04, 14.05, 14.06, 14.07, 14.08,
       14.09, 14.1 , 14.11, 14.12, 14.13, 14.14, 14.15, 14.16, 14.17,
       14.18, 14.19, 14.2 , 14.21, 14.22, 14.23, 14.24, 14.25, 14.26,
       14.27, 14.28, 14.29, 14.3 , 14.31, 14.32, 14.33, 14.34, 14.35,
       14.36, 14.37, 14.38, 14.39, 14.4 , 14.41, 14.42, 14.43, 14.44,
       14.45, 14.46, 14.47, 14.48, 14.49, 14.5 , 14.51, 14.52, 14.53,
       14.54, 14.55, 14.56, 14.57, 14.58, 14.59, 14.6 , 14.61, 14.62,
       14.63, 14.64, 14.65, 14.66, 14.67, 14.68, 14.69, 14.7 , 14.71,
       14.72, 14.73, 14.74, 14.75, 14.76, 14.77, 14.78, 14.79, 14.8 ,
       14.81, 14.82, 14.83, 14.84, 14.85, 14.86, 14.87, 14.88, 14.89,
       14.9 , 14.91, 14.92, 14.93, 14.94, 14.95, 14.96, 14.97, 14.98,
       14.99, 15.  , 15.01, 15.02, 15.03, 15.04, 15.05, 15.06, 15.07,
       15.08, 15.09, 15.1 , 15.11, 15.12, 15.13, 15.14, 15.15, 15.16,
       15.17, 15.18, 15.19, 15.2 , 15.21, 15.22, 15.23, 15.24, 15.25,
       15.26, 15.27,

In [42]:
A = np.array([[5.6,6.6],[7.6,8.6]])
b = np.array([4, 5, 6, 7, 8])

np.array(list(map(lambda x: np.argmin(np.abs(b-x)), A.flatten()))).reshape(A.shape)
#list(map(lambda x: b-x, A.flatten()))

array([[2, 3],
       [4, 4]])

In [6]:
from gymnasium import spaces

observation_space = gym.spaces.Dict(
            {
                "x": gym.spaces.Box(0, 200, shape=(4,), dtype=np.int64),
                "y": gym.spaces.Box(0, 200, shape=(4,), dtype=np.int64),
                "dz": gym.spaces.Box(11, 20, shape=(4,), dtype=np.float64),
                "df": gym.spaces.Box(-np.inf, np.inf, shape=(4,), dtype=np.float64),
            }
        )
print(observation_space.sample())

{'df': array([-1.5477269 , -0.33449369, -1.02469609,  0.50257519]), 'dz': array([16.13904256, 11.13598814, 14.49091333, 16.7682955 ]), 'x': array([143,  56, 162, 184]), 'y': array([156, 150, 177,  59])}


In [ ]:
# {
#                 "x": gym.spaces.Box(0, x_px_max, shape=(num_historic_data,), dtype=np.int64),
#                 "y": gym.spaces.Box(0, y_px_max, shape=(num_historic_data,), dtype=np.int64),
#                 "dz": gym.spaces.Box(self.z_min, self.z_max, shape=(num_historic_data,), dtype=np.float64),
#                 "df": gym.spaces.Box(-np.inf, np.inf, shape=(num_historic_data,), dtype=np.float64),
#             }